## This pipeline identifies points where coordinated hype (sentiment) and unusual price movement (returns) overlap. These overlap helps to detect the pump-and-dump activities

## Step 1 Loaded the pretrained Models from previous steps

In [9]:
import pickle

with open("absa_model.pkl", "rb") as f:
    absa_model = pickle.load(f)

with open("emotion_model.pkl", "rb") as f:
    emotion_model = pickle.load(f)

print("Models loaded successfully")


Models loaded successfully


## Collected raw tweets with timestamps

In [10]:
import pandas as pd
tweets_df=pd.read_csv("adani_tweets.csv")

## Step 2-: ABSA inference 

In [12]:
tweets_df[["management", "governance", "fundamentals", "hype", "other"]] = absa_model.predict(tweets_df["tweet_text"])
print("ABSA features extracted")


ABSA features extracted


In [13]:
tweets_df["emotion"] = emotion_model.predict(tweets_df["tweet_text"])
print("Emotion extracted")

Emotion extracted


In [14]:
emotion_dummies = pd.get_dummies(tweets_df["emotion"], prefix="emo")
tweets_features = pd.concat(
    [tweets_df[["management", "governance", "fundamentals", "hype", "other"]],
     emotion_dummies],
    axis=1
)

## Step 3-: Clustering to findout the coordinated attempts for Pump and Dump

In [16]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(tweets_features)

kmeans = KMeans(n_clusters=4, random_state=42)
tweets_df["cluster"] = kmeans.fit_predict(X_scaled)

print("Clusters formed, showing coordinated groups")


Clusters formed, showing coordinated groups


In [19]:
with open("sentiment_forecast_model.pkl", "rb") as f:
    forecast_model = pickle.load(f)

tweets_df["timestamp"] = pd.to_datetime(tweets_df["timestamp"])
sentiment_time = tweets_df.groupby("timestamp")[["management", "governance", "fundamentals", "hype", "other"]].mean()

forecast_df = forecast_model.predict(sentiment_time)


In [20]:
sentiment_time["forecasted"] = forecast_df["predicted"]
sentiment_time["drift"] = sentiment_time["management"] - sentiment_time["forecasted"]
sentiment_time["anomaly"] = abs(sentiment_time["drift"]) > 0.2


## Step 4-: Residual Based Probablistic price prediction interval

In [39]:
import pandas as pd

In [40]:
prices_df=pd.read_csv("adani_prices.csv")

In [ ]:
merged_df = pd.concat([prices_df.reset_index(drop=True), tweets_features.reset_index(drop=True)], axis=1)

In [ ]:
merged_df["next_day_return"] = merged_df["close"].shift(-1) / merged_df["close"] - 1
merged_df = merged_df[:-1]


In [ ]:
import pickle
pickle.dump(price_model, open("price_prediction_model.pkl", "wb"))

merged_df["predicted_return"] = price_model.predict(X)


In [34]:
merged_df["residual"] = merged_df["next_day_return"] - merged_df["predicted_return"]


In [35]:
window = 20  
merged_df["rolling_sigma"] = merged_df["residual"].rolling(window).std()
merged_df["rolling_sigma"].fillna(merged_df["rolling_sigma"].mean(), inplace=True)


C:\Users\aruki\AppData\Local\Temp\ipykernel_11976\4235677166.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_df["rolling_sigma"].fillna(merged_df["rolling_sigma"].mean(), inplace=True)


In [ ]:
k = 2  

merged_df["upper_bound"] = merged_df["predicted_return"] + k * merged_df["rolling_sigma"]
merged_df["lower_bound"] = merged_df["predicted_return"] - k * merged_df["rolling_sigma"]

merged_df["anomaly_flag"] = ((merged_df["next_day_return"] > merged_df["upper_bound"]) |
                             (merged_df["next_day_return"] < merged_df["lower_bound"])).astype(int)

print("Price anomaly detection complete")



Price anomaly detection complete


## Step 5-: Final Anomaly Flag based on the overlap

In [ ]:
merged_df["manipulation_flag"] = (
    (merged_df["sentiment_anomaly"] == 1) & 
    (merged_df["price_anomaly_flag"] == 1)
).astype(int)


print("Anomaly detected from both sides: Possible pump-and-dump")

 Anomaly detected from both sides: Possible pump-and-dump
